# Lab Day 1 — Mạng nơ-ron trên Forest CoverType

**MSSV:** 2A202602630

Notebook chạy từ đầu đến cuối (*Restart & Run All*). Mọi thí nghiệm gọi **một** hàm `run_experiment(cfg, data)`; mỗi thí nghiệm chỉ đổi dict cấu hình.

- Code: `data.py`, `model.py`, `optimizer.py`, `train.py`, `plots.py`, `results_table.py` (cùng thư mục với notebook).
- Kết quả: `../results/<exp_id>.json`, ảnh `../figures/<exp_id>.png`, bảng `../experiments.xlsx`, dự đoán `../predictions_eval.csv`.
- Mọi lựa chọn (lr, epoch tốt nhất, cấu hình cuối) **chỉ dựa trên val**. Tập eval chỉ được dùng ở Part 4, đúng hai lần: baseline và cấu hình cuối cùng.

**Chạy trên Google Colab:** *Runtime → Change runtime type → T4 GPU*, rồi *Runtime → Run all*. Ô đầu tiên tự mount Google Drive,
clone repo vào `MyDrive/K4-Track4-Day1-Neuralnetwork` (nếu chưa có) và chuyển vào thư mục code. Kết quả nằm trên Drive nên không mất khi Colab ngắt kết nối;
chạy lại thì các thí nghiệm đã xong được đọc lại từ `results/` thay vì huấn luyện lại.

In [ ]:
# ===== Môi trường và đường dẫn =====
import os, sys, json, math, subprocess

IN_COLAB = "google.colab" in sys.modules
FORK_URL = "https://github.com/hieulovecat/K4-Track4-Day1-Neuralnetwork.git"
SUBMISSION = "submission_2A202602630"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    colab_root = "/content/drive/MyDrive/K4-Track4-Day1-Neuralnetwork"
    if not os.path.exists(colab_root):
        subprocess.run(["git", "clone", FORK_URL, colab_root], check=True)
    os.chdir(f"{colab_root}/{SUBMISSION}/code")
sys.path.insert(0, os.getcwd())  # để import data.py, model.py, ... ở thư mục hiện tại

REPO_ROOT = "../.."                                  # chứa data/, scripts/, templates/
OUT_DIR = os.environ.get("LAB_OUT_DIR", "..")        # = submission_2A202602630/
FAST = os.environ.get("LAB_FAST") == "1"             # chỉ để thử nhanh pipeline (1 epoch); KHÔNG dùng cho bài nộp
RERUN = False                                        # True = huấn luyện lại mọi thí nghiệm, bỏ qua results/ đã lưu
EPOCHS = 1 if FAST else 20                           # cùng số epoch cho mọi thí nghiệm (trừ cấu hình cuối)

FIG_DIR, RES_DIR = f"{OUT_DIR}/figures", f"{OUT_DIR}/results"
CKPT_DIR = f"{OUT_DIR}/_ckpt" if FAST else f"{REPO_ROOT}/checkpoints"   # trọng số: NGOÀI thư mục nộp
for d in (FIG_DIR, RES_DIR, CKPT_DIR):
    os.makedirs(d, exist_ok=True)
UTF8_ENV = {**os.environ, "PYTHONIOENCODING": "utf-8"}
print("cwd     :", os.getcwd())
print("OUT_DIR :", os.path.abspath(OUT_DIR), "| FAST =", FAST, "| EPOCHS =", EPOCHS)

In [ ]:
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import Image, display

from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval, per_class_scores, confusion_matrix
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx, jsonable

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("torch", torch.__version__, "| device:", device,
      "|", torch.cuda.get_device_name(0) if device == "cuda" else "")
if device == "cuda":
    print("bf16 hỗ trợ:", torch.cuda.is_bf16_supported())

## Part 0 — Dữ liệu
Chia train/eval bằng `scripts/split_data.py` (theo `split_metadata.csv`, không tự chia lại).
Sau đó tách 20% train làm validation (phân tầng, seed 42), chuẩn hoá 10 cột số bằng mean/std **chỉ của phần train còn lại**,
rồi đưa toàn bộ lên device một lần.

In [ ]:
proc = subprocess.run([sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT,
                      capture_output=True, text=True, encoding="utf-8", env=UTF8_ENV)
print(proc.stdout or proc.stderr)
assert proc.returncode == 0

data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=f"{REPO_ROOT}/data/processed")
assert len(data["X_tr"]) == 371_847 and len(data["X_val"]) == 92_962 and len(data["X_eval"]) == 116_203

**Nhận xét Part 0:** tỉ lệ lớp ở train/val/eval gần như bằng nhau (phân tầng). Mốc thấp nhất là "luôn đoán lớp 1": accuracy val 0,4876.
mean/std chỉ tính trên train: nếu tính trên val/eval, phép chuẩn hoá đã "nhìn" dữ liệu dùng để đánh giá (rò rỉ thông tin).

## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
`M-base` = `54 → 256 → 128 → 7`, ReLU, khởi tạo He, **47 879 tham số**, đầu ra logits `(B, 7)` (softmax nằm trong `F.cross_entropy`).

In [ ]:
set_seed(1)
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
assert count_params(model) == EXPECTED_PARAMS[(256, 128)] == 47_879
print(model)
print("số tham số:", count_params(model))

# shape sau từng lớp với một lô ngẫu nhiên (8, 54)
h = torch.randn(8, 54, device=device)
for layer in model.net:
    h = layer(h)
    print(f"{layer.__class__.__name__:8s} -> {tuple(h.shape)}")
assert h.shape == (8, 7)

In [ ]:
# Phép thử 1: loss bước 0 trên val (eval mode) so với ln 7
step0 = evaluate(model, data["X_val"], data["y_val"])
with torch.no_grad():
    logits0 = model(data["X_val"][:8192])
print(f"loss bước 0 = {step0['loss']:.4f}   ln 7 = {math.log(7):.4f}   chênh = {step0['loss'] - math.log(7):+.4f}")
print(f"std của logits ở bước 0 = {logits0.std().item():.3f}")
print("loss bước 0 theo từng cách khởi tạo (cùng seed 1):")
for init in ["he", "default", "xavier", "normal", "zeros"]:
    set_seed(1)
    m = MLP(init=init).to(device)
    print(f"  {init:8s} {evaluate(m, data['X_val'], data['y_val'])['loss']:.4f}")

**Nhận xét phép thử 1:** loss bước 0 với He lớn hơn ln 7 khoảng 0,2–0,4 (tuỳ seed). Không phải lỗi: He giữ phương sai kích hoạt qua các lớp ReLU,
nên logits ban đầu có độ lệch chuẩn khoảng 0,6 thay vì ≈ 0. Softmax của các logit ngẫu nhiên đó không đều, mô hình "tự tin sai" một chút,
nên CE > ln 7. Với `normal(0, 0.01)` hay `zeros` logits ≈ 0 nên loss ≈ ln 7 đúng như lý thuyết. Chênh lệch nhỏ (< 0,5) nên vẫn
coi là khởi tạo hợp lý; một khởi tạo hỏng (ví dụ std quá lớn) sẽ cho loss bước 0 lớn hơn nhiều.

In [ ]:
# Phép thử 2: quá khớp 20 mẫu (dropout tắt). Loss phải về gần 0.
set_seed(0)
tiny = MLP(init="he").to(device)
x20, y20 = data["X_tr"][:20], data["y_tr"][:20]
opt = torch.optim.Adam(tiny.parameters(), lr=1e-2)
curve = []
for step in range(300):
    tiny.train()
    loss = F.cross_entropy(tiny(x20), y20)
    opt.zero_grad(); loss.backward(); opt.step()
    curve.append(loss.item())
acc20 = (predict(tiny, x20) == y20).float().mean().item()
print(f"loss cuối = {curve[-1]:.2e}, accuracy trên 20 mẫu = {acc20:.2f}")
assert curve[-1] < 1e-2 and acc20 == 1.0

plt.figure(figsize=(5, 3))
plt.semilogy(curve)
plt.xlabel("bước"); plt.ylabel("train loss (log)"); plt.title("Quá khớp 20 mẫu (Adam lr=1e-2)")
plt.grid(alpha=0.3); plt.show()

In [ ]:
# Gradient có chảy tới mọi tham số không?
set_seed(1)
m = MLP(init="he").to(device)
xb, yb = data["X_tr"][:512], data["y_tr"][:512]
F.cross_entropy(m(xb), yb).backward()
for name, p in m.named_parameters():
    g = p.grad
    print(f"{name:14s} shape {str(tuple(p.shape)):12s} ‖grad‖ = {g.norm().item():.4e}")
    assert g is not None and g.norm().item() > 0
print("mọi tham số đều có gradient khác 0")

**Nhận xét Part 1:** đúng 47 879 tham số, logits `(B, 7)`. Mô hình học thuộc được 20 mẫu (loss → gần 0, accuracy 100%), nên vòng
forward/backward/cập nhật, nhãn và hàm mất mát không có lỗi. Mọi lớp `W1, b1, W2, b2, W3, b3` đều nhận gradient khác 0.

## Part 2 — Pipeline và baseline
Baseline: `M-base`, He, CE, SGD+momentum 0,9, batch 512, 20 epoch, không dropout/clip, FP32.

Hàm `run()` dưới đây bọc `run_experiment`: lưu `results/<exp_id>.json` và `figures/<exp_id>.png`.
Nếu `results/<exp_id>.json` đã có **với đúng cấu hình đó**, nó đọc lại kết quả thay vì huấn luyện lại (để chạy tiếp được sau khi Colab ngắt).
Trọng số của epoch tốt nhất chỉ lưu (ra `checkpoints/`, ngoài thư mục nộp) cho các cấu hình cần đánh giá trên eval.

In [ ]:
RESULTS = {}   # exp_id -> result, theo thứ tự chạy; mỗi phần tử là một dòng của experiments.xlsx


def run(cfg, need_state=False, show=False):
    # mặc định = baseline: lr = BASE_LR (chọn ở 2a bằng val), EPOCHS epoch; cfg chỉ ghi yếu tố bị đổi
    cfg = {**DEFAULT_CFG, "epochs": EPOCHS, "lr": globals().get("BASE_LR"), **cfg}
    exp_id = cfg["exp_id"]
    res_path, ckpt_path = f"{RES_DIR}/{exp_id}.json", f"{CKPT_DIR}/{exp_id}.pt"
    cached = None
    if not RERUN and os.path.exists(res_path) and (not need_state or os.path.exists(ckpt_path)):
        with open(res_path, encoding="utf-8") as f:
            cached = json.load(f)
        if cached["cfg"] != jsonable(cfg):   # cấu hình đã đổi -> chạy lại
            cached = None
    if cached is not None:
        r = cached
        r["cfg"]["hidden"] = tuple(r["cfg"]["hidden"])
        if need_state:
            r["best_state"] = torch.load(ckpt_path, map_location=device)
        s = r["summary"]
        print(f"[{exp_id}] (đọc lại từ results/) best epoch {s['best_epoch']}: val_loss {s['best_val_loss']}  "
              f"val_acc {s['val_acc']}  val_macro_f1 {s['val_macro_f1']}")
    else:
        r = run_experiment(cfg, data)
        save_result(r, RES_DIR)
        if need_state and r["best_state"] is not None:
            torch.save(r["best_state"], ckpt_path)
    fig_path = f"{FIG_DIR}/{exp_id}.png"
    plot_run(r, fig_path)
    if show:
        display(Image(fig_path))
    RESULTS[exp_id] = r
    return r


def f1(exp_id):
    v = RESULTS[exp_id]["summary"]["val_macro_f1"]
    return -1.0 if v is None or (isinstance(v, float) and math.isnan(v)) else v


def table(ids):
    # bảng so sánh với baseline: Δ val macro-F1 so với trung bình baseline, có vượt 2σ không
    rows = []
    for i in ids:
        s, c = RESULTS[i]["summary"], RESULTS[i]["cfg"]
        d = None if f1(i) < 0 else f1(i) - BASE_F1_MEAN
        rows.append(dict(exp_id=i, description=c["description"], val_acc=s["val_acc"], val_macro_f1=s["val_macro_f1"],
                         delta_vs_base=d, beyond_2sigma=None if d is None else ("Có" if abs(d) > NOISE else "Không"),
                         best_epoch=s["best_epoch"], gap_val_train=(s["final_val_loss"] or np.nan) - (s["final_train_loss"] or np.nan),
                         s_per_epoch=s["time_per_epoch_s"], diverged=s["diverged"]))
    return pd.DataFrame(rows).set_index("exp_id").round(4)


def compare(group, ids, metrics=("val_loss", "val_macro_f1", "grad_norm"), title=""):
    path = f"{FIG_DIR}/compare_{group}.png"
    plot_compare([RESULTS[i] for i in ids], list(metrics), path, title or f"So sánh nhóm {group}")
    display(Image(path))

### 2a. Chọn lr cho baseline (bằng val)
Thử `lr ∈ {0.01, 0.03, 0.1}` với SGD+momentum, seed 1, 20 epoch; chọn lr có val macro-F1 cao nhất.
Ba lần chạy này cũng là các dòng của nhóm `optimizer` (SGD+momentum ở 3 lr).

In [ ]:
SGDM_LRS = [0.01, 0.03, 0.1]
for lr in SGDM_LRS:
    run(dict(exp_id=f"opt-sgdm-lr{lr:g}", group="optimizer", lr=lr,
             description=f"SGD+momentum 0.9, lr={lr:g} (dò lr cho baseline)"))
lr_scores = {lr: f1(f"opt-sgdm-lr{lr:g}") for lr in SGDM_LRS}
BASE_LR = max(lr_scores, key=lr_scores.get)
print("val macro-F1 theo lr:", {k: round(v, 4) for k, v in lr_scores.items()}, "-> BASE_LR =", BASE_LR)

### 2b. Baseline với 3 seed (đo độ nhiễu)

In [ ]:
BASE_IDS = [f"base-s{s}" for s in (1, 2, 3)]
for s, i in zip((1, 2, 3), BASE_IDS):
    run(dict(exp_id=i, group="baseline", lr=BASE_LR, seed=s,
             description=f"Baseline M-base, SGD+momentum lr={BASE_LR:g}, seed {s}"),
        need_state=(s == 1), show=(s == 1))

base_f1 = np.array([f1(i) for i in BASE_IDS])
base_acc = np.array([RESULTS[i]["summary"]["val_acc"] for i in BASE_IDS])
BASE_F1_MEAN, BASE_F1_STD = base_f1.mean(), base_f1.std(ddof=1)
NOISE = 2 * BASE_F1_STD
print(f"val macro-F1: {BASE_F1_MEAN:.4f} ± {BASE_F1_STD:.4f}  -> ngưỡng nhiễu 2σ = {NOISE:.4f}")
print(f"val accuracy: {base_acc.mean():.4f} ± {base_acc.std(ddof=1):.4f}")
assert base_acc.min() > 0.4876, "baseline không vượt mốc đoán lớp đa số -> có lỗi"
table(BASE_IDS)

**Nhận xét baseline (viết sau khi chạy):** hình dạng đường cong train/val (còn giảm ở epoch 20? best epoch là epoch cuối?
khoảng cách val − train?), val accuracy so với mốc 0,4876, độ nhiễu giữa 3 seed (σ và 2σ ở trên). Từ đây, mọi chênh lệch val macro-F1
nhỏ hơn 2σ đều **chưa** là bằng chứng.

## Part 3 — Thí nghiệm
Mỗi thí nghiệm đổi **một** yếu tố so với `base-s1` (cùng seed 1, cùng split, 20 epoch). Dự đoán viết **trước** khi chạy; đối chiếu viết sau.
Cột `beyond_2sigma` so chênh lệch val macro-F1 với ngưỡng nhiễu 2σ của baseline.

### Chủ đề 1 — Hàm mất mát: CE vs MSE (nhóm `loss`)
MSE tính như `nn.MSELoss`: trung bình trên mọi phần tử `(B·7)` giữa logit thô và one-hot, không có hệ số 1/2.

**Dự đoán:** MSE học chậm hơn và macro-F1 thấp hơn CE ở cùng lr. Gradient của CE theo logit là `p − y`, không bão hoà khi dự đoán sai nặng;
gradient MSE theo logit là `2(z − y)/7` (do lấy trung bình trên 7 cột) nên nhỏ hơn nhiều và không gắn với xác suất. Lớp hiếm bị ảnh hưởng nhiều nhất.
Thử thêm lr ×3 để xem chênh lệch có phải chỉ do độ lớn gradient. So bằng accuracy/macro-F1, không so loss (khác thang đo).

In [ ]:
LOSS_IDS = []
for mult in (1, 3):
    lr = BASE_LR * mult
    i = f"loss-mse-lr{lr:g}"
    run(dict(exp_id=i, group="loss", loss="mse", lr=lr, description=f"MSE trên one-hot thay CE, lr={lr:g}"))
    LOSS_IDS.append(i)
compare("loss", ["base-s1"] + LOSS_IDS, metrics=("val_acc", "val_macro_f1", "grad_norm"))
table(["base-s1"] + LOSS_IDS)

**Đối chiếu (viết sau khi chạy):** khớp/khác dự đoán? macro-F1 của MSE so với CE, có vượt 2σ? lr ×3 có thu hẹp khoảng cách không? grad_norm của MSE nhỏ hơn bao nhiêu lần?

### Chủ đề 2 — Bộ tối ưu hoá (nhóm `optimizer`)
Mỗi bộ tối ưu thử ≥ 2 lr, so ở lr tốt nhất của nó. SGD+momentum đã có 3 lr ở Part 2a.
SGD thuần: lr lớn hơn (momentum 0,9 tương đương nhân bước hiệu dụng ×10). Adam/AdamW: `betas=(0.9, 0.999)`, `eps=1e-8`; AdamW `weight_decay=0.01`.

**Dự đoán:** SGD thuần ở cùng số epoch sẽ kém SGD+momentum nếu lr nhỏ; với lr ×10 thì gần bằng.
Adam hội tụ nhanh hơn ở các epoch đầu (bước chia theo √v̂ từng tham số, ít nhạy với độ lớn gradient), ở lr tốt nhất có thể nhỉnh hơn SGD+momentum
nhưng chênh lệch có thể không vượt 2σ. AdamW với wd nhỏ gần như giống Adam vì mô hình chưa quá khớp.

In [ ]:
OPT_IDS = [f"opt-sgdm-lr{lr:g}" for lr in SGDM_LRS]
for lr in (BASE_LR * 3, BASE_LR * 10):
    i = f"opt-sgd-lr{lr:g}"
    run(dict(exp_id=i, group="optimizer", optimizer="sgd", lr=lr, description=f"SGD thuần (không momentum), lr={lr:g}"))
    OPT_IDS.append(i)
for lr in (3e-4, 1e-3, 3e-3):
    i = f"opt-adam-lr{lr:g}"
    run(dict(exp_id=i, group="optimizer", optimizer="adam", lr=lr, description=f"Adam, lr={lr:g}"))
    OPT_IDS.append(i)
for lr in (1e-3, 3e-3):
    i = f"opt-adamw-lr{lr:g}-wd0.01"
    run(dict(exp_id=i, group="optimizer", optimizer="adamw", lr=lr, weight_decay=0.01,
             description=f"AdamW, lr={lr:g}, weight_decay=0.01"))
    OPT_IDS.append(i)

best_per_opt = {}
for i in OPT_IDS:
    name = RESULTS[i]["cfg"]["optimizer"]
    if name not in best_per_opt or f1(i) > f1(best_per_opt[name]):
        best_per_opt[name] = i
print("lr tốt nhất (theo val macro-F1) của mỗi bộ tối ưu:", best_per_opt)
compare("optimizer", list(best_per_opt.values()), title="Mỗi bộ tối ưu ở lr tốt nhất của nó (theo val)")
table(OPT_IDS)

**Đối chiếu (viết sau khi chạy):** bảng mỗi bộ tối ưu ở lr tốt nhất; bộ nào nhạy với lr hơn; chênh lệch giữa các bộ ở lr tốt nhất có vượt 2σ không; cơ chế (momentum, chuẩn hoá bước của Adam).

### Chủ đề 3 — Hyper-parameter (nhóm `hparam`)
- Batch 128 và 2048 (cùng lr): cùng 20 epoch nhưng số bước cập nhật lần lượt ×4 và ÷4 so với batch 512.
- Batch 2048 với lr ×4 + khởi động 1 epoch (quy tắc "lô ×k thì η ×k"; đổi 3 thứ cùng lúc, có ghi rõ).
- `M-wide` (512-256, 161 287 tham số) và `M-deep` (256-128-64, 55 687 tham số).

**Dự đoán:** batch 2048 cùng lr kém rõ (ít bước hơn 4 lần); tăng lr ×4 kéo lại phần lớn. Batch 128 tốt hơn một chút nhưng chậm hơn mỗi epoch.
Mô hình rộng/sâu hơn có val macro-F1 cao hơn vì baseline ở 20 epoch chưa quá khớp (bài toán cần nhiều năng lực để phân biệt các lớp hiếm).

In [ ]:
HP_IDS = []
for b in (128, 2048):
    i = f"hp-batch{b}"
    run(dict(exp_id=i, group="hparam", batch=b, lr=BASE_LR, description=f"batch {b} (lr giữ nguyên {BASE_LR:g})"))
    HP_IDS.append(i)
steps_2048 = math.ceil(len(data["X_tr"]) / 2048)
i = f"hp-batch2048-lr{BASE_LR * 4:g}-warmup"
run(dict(exp_id=i, group="hparam", batch=2048, lr=BASE_LR * 4, scheduler="warmup", warmup_steps=steps_2048,
         description=f"batch 2048, lr ×4 = {BASE_LR * 4:g}, khởi động tuyến tính 1 epoch ({steps_2048} bước)"))
HP_IDS.append(i)
for name, hidden in (("wide", (512, 256)), ("deep", (256, 128, 64))):
    i = f"hp-{name}"
    run(dict(exp_id=i, group="hparam", hidden=hidden, description=f"M-{name}: hidden={'-'.join(map(str, hidden))}"))
    HP_IDS.append(i)
compare("hparam", ["base-s1"] + HP_IDS)
table(["base-s1"] + HP_IDS)

**Đối chiếu (viết sau khi chạy):** giải thích bằng số bước cập nhật (727 / 2 906 / 182 bước mỗi epoch) và thời gian mỗi epoch; quy tắc tăng lr theo lô có bù được không; rộng/sâu hơn giúp bao nhiêu so với 2σ.

### Chủ đề 4 — Dropout (nhóm `dropout`)
q ∈ {0,1; 0,3}, đặt sau ReLU của hai lớp ẩn. train loss đo ở `eval()` nên so sánh được với val loss.

**Dự đoán:** baseline 20 epoch có khoảng cách val − train loss rất nhỏ (chưa quá khớp, 370k mẫu so với 48k tham số), nên dropout
không có gì để chữa: nó thêm nhiễu và giảm năng lực hiệu dụng, val macro-F1 giảm, q = 0,3 giảm nhiều hơn q = 0,1.

In [ ]:
DROP_IDS = []
for q in (0.1, 0.3):
    i = f"drop-{q:g}"
    run(dict(exp_id=i, group="dropout", dropout=q, description=f"dropout q={q:g} sau mỗi ReLU ẩn"))
    DROP_IDS.append(i)
compare("dropout", ["base-s1"] + DROP_IDS, metrics=("train_loss", "val_loss", "val_macro_f1"))
table(["base-s1"] + DROP_IDS)

**Đối chiếu (viết sau khi chạy):** khoảng cách val − train của baseline bao nhiêu (có quá khớp không)? dropout thay đổi khoảng cách đó và val macro-F1 thế nào?

### Chủ đề 5 — Gradient clipping (nhóm `clipping`)
`g ← g · min(1, c/‖g‖)` với chuẩn L2 toàn cục. **c chọn từ `grad_norm` của baseline:** c = trung vị ‖g‖ của `base-s1`, để clip thực sự
kích hoạt ở khoảng một nửa số bước.
Thí nghiệm phản chứng: tăng lr (×10, ×30, ×100) cho tới khi huấn luyện không clip hỏng (NaN hoặc macro-F1 sụt mạnh), rồi chạy lại đúng lr đó có clip.

**Dự đoán:** ở lr bình thường, clip với c = trung vị gần như không đổi kết quả (chỉ thu nhỏ bước khi gradient lớn; SGD+momentum vẫn đi đúng hướng).
Ở lr cao, không clip thì loss nổ; có clip thì độ dài mỗi bước bị chặn ở lr·c nên huấn luyện ổn định hơn, dù chưa chắc bằng baseline.

In [ ]:
s1 = RESULTS["base-s1"]["summary"]
C = float(f"{s1['grad_norm_median']:.2g}")
print(f"grad_norm của base-s1: trung vị {s1['grad_norm_median']:.3f}, p90 {s1['grad_norm_p90']:.3f} -> c = {C:g}")

CLIP_IDS = [f"clip-c{C:g}"]
run(dict(exp_id=CLIP_IDS[0], group="clipping", lr=BASE_LR, clip_norm=C, description=f"clip c={C:g} (trung vị grad_norm baseline), lr={BASE_LR:g}"))

HIGH_LR = None
for mult in (10, 30, 100):
    lr = BASE_LR * mult
    i = f"clip-none-lr{lr:g}"
    r = run(dict(exp_id=i, group="clipping", lr=lr, description=f"KHÔNG clip, lr cao = {lr:g} (×{mult})"))
    CLIP_IDS.append(i)
    if r["summary"]["diverged"] or f1(i) < BASE_F1_MEAN - 0.1:
        HIGH_LR = lr
        break
if HIGH_LR is None:
    print("Không lr nào trong ×10..×100 làm hỏng huấn luyện; dùng lr ×100 cho phép thử có clip.")
    HIGH_LR = BASE_LR * 100
i = f"clip-c{C:g}-lr{HIGH_LR:g}"
run(dict(exp_id=i, group="clipping", lr=HIGH_LR, clip_norm=C, description=f"clip c={C:g} ở lr cao = {HIGH_LR:g}"))
CLIP_IDS.append(i)
for i in CLIP_IDS:
    print(f"{i:24s} tỉ lệ bước bị clip = {RESULTS[i]['summary']['clip_frac']:.2%}")
compare("clipping", ["base-s1"] + CLIP_IDS, metrics=("val_loss", "val_macro_f1", "grad_norm_max"))
table(["base-s1"] + CLIP_IDS)

**Đối chiếu (viết sau khi chạy):** ở lr thường clip kích hoạt bao nhiêu % bước và có đổi kết quả vượt 2σ không; ở lr cao, không clip thì grad_norm có "gai" thế nào, có clip thì sao?

### Chủ đề 6 — Mixed precision (nhóm `amp`)
FP16: `autocast(float16)` + `GradScaler` (unscale trước khi đo/cắt gradient). BF16: `autocast(bfloat16)`, không GradScaler.
Tham số vẫn FP32; chỉ phép toán trong forward hạ độ chính xác. Cần GPU CUDA.

**Dự đoán:** độ chính xác gần như không đổi (trong 2σ). Mô hình rất nhỏ (48k tham số, lô 512) nên thời gian bị chi phối bởi chi phí gọi kernel,
mixed precision khó nhanh hơn, có thể còn chậm hơn vì thêm phép ép kiểu và GradScaler. BF16 trên T4 không có phần cứng hỗ trợ nên có thể chậm.
FP16 cần nhân loss với hệ số s vì khoảng biểu diễn hẹp (nhỏ nhất ~6e-8, gradient nhỏ bị làm tròn về 0);
BF16 có cùng 8 bit mũ như FP32 nên không bị underflow, chỉ kém chính xác phần định trị.

In [ ]:
AMP_IDS = []
if device == "cuda":
    for prec in ("fp16", "bf16"):
        i = f"amp-{prec}"
        run(dict(exp_id=i, group="amp", precision=prec, description=f"mixed precision {prec}"
                 + (" + GradScaler" if prec == "fp16" else "")))
        AMP_IDS.append(i)
    rows = []
    for i in ["base-s1"] + AMP_IDS:
        s = RESULTS[i]["summary"]
        rows.append(dict(exp_id=i, s_per_epoch=s["time_per_epoch_s"], peak_mem_MB=s["peak_mem_MB"],
                         val_macro_f1=s["val_macro_f1"], amp_skipped_steps=s["amp_skipped_steps"]))
    display(pd.DataFrame(rows).set_index("exp_id").round(4))
    compare("amp", ["base-s1"] + AMP_IDS, metrics=("val_loss", "val_macro_f1", "epoch_time_s"))
else:
    print("Không có GPU CUDA: bỏ qua mixed precision (chạy notebook trên Colab GPU để có chủ đề này).")

**Đối chiếu (viết sau khi chạy):** thời gian/epoch, bộ nhớ cực đại và macro-F1 của FP32/FP16/BF16; chỉ nói "nhanh hơn" nếu số đo cho thấy vậy; giải thích.

### Chủ đề 7 — Khởi tạo tham số (nhóm `init`)
`zeros` (W = 0), `normal` (N(0, 0,01²)), `xavier` (`xavier_normal_`, Var = 2/(n_in + n_out)), `he` (baseline, Var = 2/n_in), `default` (mặc định `nn.Linear`). Bias = 0 (trừ `default`).

**Dự đoán:** `zeros`: mọi nơ-ron trong một lớp giống hệt nhau và ReLU(0) = 0 nên gradient của W1, W2 bằng 0, chỉ bias lớp cuối học được →
mô hình chỉ học được tần suất lớp, kẹt ở mức "đoán lớp đa số" (acc ≈ 0,4876). `normal(0,01)`: kích hoạt teo dần qua các lớp, ban đầu học chậm
nhưng 3 lớp vẫn đủ nông để hồi phục. `xavier`/`default` gần He vì mạng chỉ có 2 lớp ẩn; khác biệt rõ chỉ thấy ở mạng sâu.

In [ ]:
# std của kích hoạt sau mỗi ReLU (và của logits) ở bước 0, trên một lô val 4096 mẫu
xb = data["X_val"][:4096]
rows = {}
for init in ["zeros", "normal", "xavier", "default", "he"]:
    set_seed(1)
    m = MLP(init=init).to(device)
    stds = activation_stats(m, xb)
    rows[init] = dict(relu1=stds[0], relu2=stds[1], logits=stds[2],
                      step0_loss=evaluate(m, data["X_val"], data["y_val"])["loss"])
display(pd.DataFrame(rows).T.round(4))

# Minh hoạ thêm (không phải thí nghiệm huấn luyện, không vào bảng): mạng 20 lớp ẩn 256
depth_stats = {}
for init in ["normal", "xavier", "default", "he"]:
    set_seed(1)
    depth_stats[init] = activation_stats(MLP(hidden=(256,) * 20, init=init).to(device), xb)[:-1]
plt.figure(figsize=(6, 3.5))
for init, s in depth_stats.items():
    plt.semilogy(range(1, 21), s, "o-", ms=3, label=init)
plt.xlabel("lớp ẩn"); plt.ylabel("std kích hoạt sau ReLU (log)"); plt.title("Mạng 20 lớp ReLU ở bước 0")
plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
INIT_IDS = []
for init in ("zeros", "normal", "xavier", "default"):
    i = f"init-{init}"
    run(dict(exp_id=i, group="init", init=init, description=f"khởi tạo {init} thay cho He"))
    INIT_IDS.append(i)
compare("init", ["base-s1"] + INIT_IDS)
table(["base-s1"] + INIT_IDS)

**Đối chiếu (viết sau khi chạy):** `zeros` có kẹt ở mức đoán đa số không (vì sao, dùng đối xứng + ReLU(0)); std kích hoạt ở bước 0 của từng cách; mạng 3 lớp có đủ sâu để thấy khác biệt không (so với hình 20 lớp).

## Part 3b — Cấu hình cuối cùng (chọn bằng val)
Ứng viên: bộ tối ưu tốt nhất theo val ở Part 3, huấn luyện 40 epoch (baseline vẫn còn giảm ở epoch 20), và cùng cấu hình đó với `M-wide`.
Chọn ứng viên có val macro-F1 cao nhất (so cả với `base-s1`), rồi chạy thêm 2 seed để đo nhiễu của cấu hình cuối.

In [ ]:
best_opt_id = max(OPT_IDS, key=f1)
bc = RESULTS[best_opt_id]["cfg"]
opt_keys = dict(optimizer=bc["optimizer"], lr=bc["lr"], weight_decay=bc["weight_decay"])
print("bộ tối ưu tốt nhất theo val:", best_opt_id, opt_keys)

FINAL_EPOCHS = 2 * EPOCHS
CANDIDATES = {
    "final-ep40": dict(hidden=(256, 128)),
    "final-wide-ep40": dict(hidden=(512, 256)),
}
for name, extra in CANDIDATES.items():
    run(dict(exp_id=f"{name}-s1", group="final", epochs=FINAL_EPOCHS, seed=1, **opt_keys, **extra,
             description=f"{best_opt_id} + hidden={'-'.join(map(str, extra['hidden']))} + {FINAL_EPOCHS} epoch"),
        need_state=True)

pool = ["base-s1"] + [f"{n}-s1" for n in CANDIDATES]
FINAL_ID = max(pool, key=f1)
print("cấu hình cuối cùng (theo val):", FINAL_ID, f"val macro-F1 = {f1(FINAL_ID):.4f}")

FINAL_SEED_IDS = [FINAL_ID]
if FINAL_ID != "base-s1":
    fc = RESULTS[FINAL_ID]["cfg"]
    for s in (2, 3):
        i = FINAL_ID.replace("-s1", f"-s{s}")
        run({**fc, "exp_id": i, "seed": s, "description": fc["description"] + f", seed {s}"})
        FINAL_SEED_IDS.append(i)
    ff = np.array([f1(i) for i in FINAL_SEED_IDS])
    print(f"cấu hình cuối, val macro-F1 qua {len(ff)} seed: {ff.mean():.4f} ± {ff.std(ddof=1):.4f}"
          f"  (baseline {BASE_F1_MEAN:.4f} ± {BASE_F1_STD:.4f})")
display(Image(f"{FIG_DIR}/{FINAL_ID}.png"))
table(pool + FINAL_SEED_IDS[1:])

## Part 4 — Đánh giá cuối trên eval, phân tích lỗi, bảng
Cấu hình đã chốt ở trên bằng val. Bây giờ mới dùng eval, **mỗi mô hình một lần**: `base-s1` và cấu hình cuối (seed 1), bằng trọng số của epoch
có val loss thấp nhất. Điểm do `scripts/evaluate.py` tính. Không quay lại chỉnh cấu hình sau bước này.

In [ ]:
EVAL_SCORES = {}
targets = {"base-s1": (f"{RES_DIR}/base-s1_predictions_eval.csv", f"{RES_DIR}/base-s1_eval_result.json")}
targets[FINAL_ID] = (f"{OUT_DIR}/predictions_eval.csv", f"{OUT_DIR}/eval_result.json")  # file nộp (ghi đè nếu FINAL_ID là base-s1)

for exp_id, (pred_path, json_path) in targets.items():
    r = RESULTS[exp_id]
    if "best_state" not in r:  # kết quả đọc lại từ results/: nạp trọng số
        r["best_state"] = torch.load(f"{CKPT_DIR}/{exp_id}.pt", map_location=device)
    print(f"===== {exp_id} =====")
    if FAST:   # chạy thử pipeline: chỉ ghi file dự đoán, KHÔNG chấm điểm eval
        final_eval(r["cfg"], r, data, pred_path)
        continue
    EVAL_SCORES[exp_id] = final_eval(r["cfg"], r, data, pred_path, repo_root=REPO_ROOT, out_json=json_path)

In [ ]:
# Phân tích lỗi theo lớp cho cấu hình cuối (số từ eval_result.json)
CLASS_NAMES = ["Spruce/Fir", "Lodgepole Pine", "Ponderosa Pine", "Cottonwood/Willow", "Aspen", "Douglas-fir", "Krummholz"]
if not FAST:
    res = EVAL_SCORES[FINAL_ID]
    per = pd.DataFrame(res["per_class"]).set_index("cls")
    per.insert(0, "tên", CLASS_NAMES)
    print(f"{FINAL_ID}: eval accuracy = {res['accuracy']:.4f}, eval macro-F1 = {res['macro_f1']:.4f}")
    b = EVAL_SCORES["base-s1"]
    print(f"base-s1 : eval accuracy = {b['accuracy']:.4f}, eval macro-F1 = {b['macro_f1']:.4f}")
    display(per.round(4))

    cm = np.array(res["confusion_matrix"])
    cm_norm = cm / cm.sum(1, keepdims=True)
    off = cm_norm.copy(); np.fill_diagonal(off, 0)
    hardest = int(per["f1"].idxmin())
    print(f"lớp khó nhất: {hardest} ({CLASS_NAMES[hardest]}), F1 = {per.loc[hardest, 'f1']:.4f}; "
          f"hay bị nhầm nhất sang lớp {off[hardest].argmax()} ({CLASS_NAMES[off[hardest].argmax()]}, "
          f"{off[hardest].max():.1%} số mẫu của lớp)")
    display(pd.DataFrame(cm, index=[f"thật {c}" for c in range(7)], columns=[f"đoán {c}" for c in range(7)]))

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    for a in range(7):
        for c in range(7):
            ax.text(c, a, f"{cm_norm[a, c]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if cm_norm[a, c] > 0.5 else "black")
    ax.set_xticks(range(7)); ax.set_yticks(range(7))
    ax.set_xlabel("dự đoán"); ax.set_ylabel("nhãn thật")
    ax.set_title(f"Ma trận nhầm lẫn chuẩn hoá theo hàng — {FINAL_ID} (eval)")
    fig.colorbar(im); plt.show()

**Phân tích lỗi (viết sau khi chạy):** lớp khó nhất là gì, nhầm với lớp nào, vì sao (số mẫu trong train; các lớp cùng vùng độ cao như
Spruce/Fir ↔ Lodgepole Pine, Ponderosa Pine ↔ Douglas-fir ↔ Cottonwood/Willow có đặc trưng địa hình chồng lấn). Val và eval có gần nhau không.

In [ ]:
# Bảng experiments.xlsx: mỗi thí nghiệm một dòng, theo thứ tự đã chạy; eval chỉ cho base-s1 và cấu hình cuối
GROUP_IDS = {"baseline": BASE_IDS, "loss": LOSS_IDS, "optimizer": OPT_IDS, "hparam": HP_IDS, "dropout": DROP_IDS,
             "clipping": CLIP_IDS, "amp": AMP_IDS, "init": INIT_IDS, "final": [i for i in RESULTS if RESULTS[i]["cfg"]["group"] == "final"]}
rows = [to_row(r, eval_scores=EVAL_SCORES.get(i)) for i, r in RESULTS.items()]

summary_notes = {}
for g, ids in GROUP_IDS.items():
    if not ids:
        continue
    best = max(ids, key=f1)
    d = f1(best) - BASE_F1_MEAN
    summary_notes[g] = (f"tốt nhất theo val: {best} (macro-F1 {f1(best):.4f}, Δ so với TB baseline {d:+.4f}, "
                        f"{'vượt' if abs(d) > NOISE else 'không vượt'} 2σ = {NOISE:.4f})")
if "baseline" in summary_notes:
    summary_notes["baseline"] = f"{len(BASE_IDS)} seed: val macro-F1 {BASE_F1_MEAN:.4f} ± {BASE_F1_STD:.4f}, 2σ = {NOISE:.4f}"

write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx",
           seed_ids=BASE_IDS, summary_notes=summary_notes)
print(f"đã ghi {OUT_DIR}/experiments.xlsx với {len(rows)} dòng")

# Kiểm tra: mỗi dòng có đúng một ảnh figures/<exp_id>.png, không có ảnh thừa
figs = {f[:-4] for f in os.listdir(FIG_DIR) if f.endswith(".png") and not f.startswith("compare_")}
ids = set(RESULTS)
print("số dòng =", len(ids), "| số ảnh =", len(figs & ids))
assert ids <= figs, f"thiếu ảnh: {ids - figs}"
if figs - ids:
    print("ẢNH THỪA (không có dòng trong bảng, nên xoá):", sorted(figs - ids))
pd.DataFrame(rows).set_index("exp_id")[["group", "lr", "val_acc", "val_macro_f1", "best_epoch", "eval_macro_f1", "diverged"]]